# Step 3B - Dev Gated Runtime Baselines

This notebook runs fair gated versions of the strong runtime interventions on `dev_tune_200` only.

It does not touch:

```text
analysis_500
ablation_500
final_test_500
final_test_full
```

It assumes Step 3A has already produced ungated dev baseline outputs and Step 3A.1 has produced:

```text
runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/
```

Goal:

```text
1. Give each runtime intervention the same subject gate where appropriate.
2. Give strongest runtime interventions the stricter subject_relation gate.
3. Report gate activation rates by bucket.
4. Build one merged dev report with old ungated and new gated methods.
5. Rank by feasible_selection_score, not raw selection_score.
```


In [ ]:
%%bash
set -euo pipefail

pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"


In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import subprocess
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "git_commit": commit,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step3b.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
print("Wrote:", out)
print(json.dumps(env, indent=2))


## Preflight

In [ ]:
from pathlib import Path
import csv
import json
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_experiment_reports.py",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/report_summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/selection_scores.csv",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated/summary.json",
]
for name in required_files:
    path = PROJECT_DIR / name
    assert path.exists(), f"Missing required file: {path}"

# Accept either the newer split Step 3A target dirs or the older combined target dir.
has_split_target = (
    PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias/summary.json"
).exists() and (
    PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support/summary.json"
).exists()
has_combined_target = (
    PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target/summary.json"
).exists()
assert has_split_target or has_combined_target, "Missing Step 3A target baseline outputs"

thresholds_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
scores_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/selection_scores.csv"
summary_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/report_summary.json"

with thresholds_path.open() as f:
    thresholds = json.load(f)
expected_selfloc = float(thresholds["selfloc_base"])
assert 0.0 < expected_selfloc < 1.0, expected_selfloc

with summary_path.open() as f:
    step3a1_summary = json.load(f)
assert step3a1_summary["primary_paraphrase_bucket"] == "declarative_paraphrases"
assert step3a1_summary["qa_format_generalization"] == "secondary_diagnostic"
assert step3a1_summary["constraint_filtering"] is True
assert step3a1_summary["analysis_500_used"] is False
assert abs(float(step3a1_summary["selfloc_base"]) - expected_selfloc) < 1e-8

with scores_path.open(newline="") as f:
    scores = list(csv.DictReader(f))
assert scores, f"No rows in {scores_path}"
for row in scores:
    got = float(row["selfloc_base"])
    assert abs(got - expected_selfloc) < 1e-8, (
        f"Broken Step 3A.1 report: expected selfloc {expected_selfloc}, got {got}"
    )

print("Python:", sys.version)
print("Project dir OK:", PROJECT_DIR)
print("Step 3A.1 fixed selfloc report OK:", expected_selfloc)
print("Step 3A target summaries:", "split" if has_split_target else "combined")


## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_target_logit_bias_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_target_support_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject_relation",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject_relation",
    project / "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject_relation",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1",
]
for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )
print("Overwrite guard passed.")


## Step 3B.1: Subject-Gated Target Logit Bias

Runs:

```text
target_logit_bias_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_target_logit_bias_subject

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_target_logit_bias_subject   --methods target_logit_bias_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 5.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_target_logit_bias_subject/stdout.log


## Step 3B.2: Subject-Gated Target Candidate Insert

Runs:

```text
target_candidate_insert_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_target_support_subject

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_target_support_subject   --methods target_candidate_insert_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_target_support_subject/stdout.log


## Step 3B.3: Subject-Gated Prompt Memory

Runs:

```text
prompt_memory_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject   --methods prompt_memory_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject/stdout.log


## Step 3B.4: Subject-Gated Bridge Controls

Runs:

```text
myopic_score_gated_subject,no_rollout_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject   --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject/stdout.log


## Step 3B.5: Subject-Gated MC Bridge

Runs:

```text
mc_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject   --methods mc_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject/stdout.log


## Step 3B.6: Subject+Relation-Gated Prompt Memory

Runs:

```text
prompt_memory_gated_subject_relation
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject_relation

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject_relation   --methods prompt_memory_gated_subject_relation   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject_relation   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject_relation/stdout.log


## Step 3B.7: Subject+Relation-Gated Bridge Controls

Runs:

```text
myopic_score_gated_subject_relation,no_rollout_bridge_gated_subject_relation
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject_relation

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject_relation   --methods myopic_score_gated_subject_relation,no_rollout_bridge_gated_subject_relation   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject_relation   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject_relation/stdout.log


## Step 3B.8: Subject+Relation-Gated MC Bridge

Runs:

```text
mc_bridge_gated_subject_relation
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject_relation

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject_relation   --methods mc_bridge_gated_subject_relation   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject_relation   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject_relation/stdout.log


## Validate Gated Run Configs And Completeness

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
method_specific = {
    "dev_tune_200_gated_target_logit_bias_subject": {
        "methods": ["target_logit_bias_gated_subject"],
        "gate_mode": "subject",
        "target_logit_bias": 5.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_target_support_subject": {
        "methods": ["target_candidate_insert_gated_subject"],
        "gate_mode": "subject",
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_prompt_memory_subject": {
        "methods": ["prompt_memory_gated_subject"],
        "gate_mode": "subject",
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_bridge_controls_subject": {
        "methods": ["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"],
        "gate_mode": "subject",
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_mc_bridge_subject": {
        "methods": ["mc_bridge_gated_subject"],
        "gate_mode": "subject",
        "target_logit_bias": 0.0,
        "mc_rollouts": 2,
    },
    "dev_tune_200_gated_prompt_memory_subject_relation": {
        "methods": ["prompt_memory_gated_subject_relation"],
        "gate_mode": "subject_relation",
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_bridge_controls_subject_relation": {
        "methods": ["myopic_score_gated_subject_relation", "no_rollout_bridge_gated_subject_relation"],
        "gate_mode": "subject_relation",
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_gated_mc_bridge_subject_relation": {
        "methods": ["mc_bridge_gated_subject_relation"],
        "gate_mode": "subject_relation",
        "target_logit_bias": 0.0,
        "mc_rollouts": 2,
    },
}

expected_protocol = {
    "protocol_version": "counterfact_direction1_v1",
    "edit_access": "given_at_edit_time",
    "training_access": "none",
    "hyperparameter_access": "dev_tune_only",
    "split_role": "dev_tune_200",
    "seed": 0,
    "model_id": "GSAI-ML/LLaDA-8B-Base",
    "eval_samples": 4,
}

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

for run_name, expected in method_specific.items():
    methods = expected["methods"]
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    cfg_path = run_dir / "run_config.json"
    rows_path = run_dir / "per_case_results.jsonl"
    summary_path = run_dir / "summary.json"
    assert cfg_path.exists(), f"Missing config: {cfg_path}"
    assert rows_path.exists(), f"Missing rows: {rows_path}"
    assert summary_path.exists(), f"Missing summary: {summary_path}"

    with cfg_path.open() as f:
        cfg = json.load(f)
    for key, value in expected_protocol.items():
        assert cfg.get(key) == value, f"{cfg_path}: expected {key}={value!r}, got {cfg.get(key)!r}"
    assert cfg["methods"] == methods, f"{cfg_path}: expected methods {methods}, got {cfg['methods']}"
    rollout = cfg.get("rollout_config", {})
    for key in ["gate_mode", "target_logit_bias", "mc_rollouts"]:
        assert rollout.get(key) == expected[key], f"{cfg_path}: expected {key}={expected[key]!r}, got {rollout.get(key)!r}"
    assert rollout.get("bridge_topk") == 4
    assert rollout.get("guidance_scale") == 1.0
    assert rollout.get("reward_mode") == "soft_overlap"
    assert rollout.get("reward_beta") == 6.0

    rows = list(read_jsonl(rows_path))
    assert rows, f"No rows in {rows_path}"
    row_methods = {row.get("method") for row in rows}
    assert row_methods == set(methods), f"{run_name}: expected {methods}, got {row_methods}"

    by_method_bucket = defaultdict(set)
    for row in rows:
        method = row.get("method")
        bucket = row.get("bucket")
        edit_id = str(row.get("edit_id") or row.get("case_id"))
        if method in methods and bucket in {"rewrite", "declarative_paraphrases", "near_locality", "far_locality"}:
            by_method_bucket[(method, bucket)].add(edit_id)
        if method in methods:
            assert row.get("gate_mode") == expected["gate_mode"], row
            assert row.get("method_variant") == method, row
            assert row.get("gate_activation_rate") is not None, row

    for method in methods:
        for bucket in ["rewrite", "declarative_paraphrases", "near_locality", "far_locality"]:
            n = len(by_method_bucket[(method, bucket)])
            assert n == 200, f"{run_name}/{method}/{bucket}: expected 200 edits, got {n}"

    print("Config and completeness OK:", run_name)

print("All Step 3B gated run configs and row completeness checks OK.")


## Build Merged Gated Report

In [ ]:
import subprocess
import sys
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1"
if out_dir.exists():
    raise RuntimeError(f"Merged report already exists: {out_dir}")

summary_candidates = [
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json",
]

split_target = [
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support/summary.json",
]
combined_target = [
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target/summary.json",
]

if all((project / path).exists() for path in split_target):
    summary_candidates.extend(split_target)
elif all((project / path).exists() for path in combined_target):
    summary_candidates.extend(combined_target)
else:
    raise RuntimeError("Missing Step 3A target baseline summaries")

summary_candidates.extend([
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_target_logit_bias_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_target_support_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject_relation/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject_relation/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject_relation/summary.json",
])

summary_paths = []
seen = set()
for rel in summary_candidates:
    path = project / rel
    if not path.exists():
        raise RuntimeError(f"Missing summary input: {path}")
    if path not in seen:
        summary_paths.append(path)
        seen.add(path)

thresholds_path = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
bootstrap_pairs = [
    "prompt_memory_gated_subject:prompt_memory",
    "myopic_score_gated_subject:myopic_score",
    "no_rollout_bridge_gated_subject:no_rollout_bridge",
    "mc_bridge_gated_subject:mc_bridge",
    "mc_bridge_gated_subject:prompt_memory_gated_subject",
    "mc_bridge_gated_subject:myopic_score_gated_subject",
    "mc_bridge_gated_subject:no_rollout_bridge_gated_subject",
    "mc_bridge_gated_subject:target_logit_bias_gated_subject",
    "mc_bridge_gated_subject_relation:prompt_memory_gated_subject_relation",
    "mc_bridge_gated_subject_relation:myopic_score_gated_subject_relation",
    "mc_bridge_gated_subject_relation:no_rollout_bridge_gated_subject_relation",
]

cmd = [
    sys.executable,
    "llada_experiment_reports.py",
    "--input",
    *[str(path.relative_to(project)) for path in summary_paths],
    "--output_dir",
    str(out_dir.relative_to(project)),
    "--thresholds_path",
    str(thresholds_path.relative_to(project)),
    "--bootstrap_baseline",
    "base",
    "--bootstrap_candidates",
    "target_logit_bias",
    "target_candidate_insert",
    "prompt_memory",
    "myopic_score",
    "no_rollout_bridge",
    "mc_bridge",
    "raw_bridge_gated_subject",
    "target_logit_bias_gated_subject",
    "target_candidate_insert_gated_subject",
    "prompt_memory_gated_subject",
    "myopic_score_gated_subject",
    "no_rollout_bridge_gated_subject",
    "mc_bridge_gated_subject",
    "prompt_memory_gated_subject_relation",
    "myopic_score_gated_subject_relation",
    "no_rollout_bridge_gated_subject_relation",
    "mc_bridge_gated_subject_relation",
    "--bootstrap_pairs",
    *bootstrap_pairs,
    "--bootstrap_metric",
    "exact_rate",
    "--bootstrap_samples",
    "2000",
    "--seed",
    "0",
]
print("Input summaries:")
for path in summary_paths:
    print(" ", path.relative_to(project))
print("Running report command")
subprocess.run(cmd, cwd=project, check=True)
print("Wrote:", out_dir)


## Inspect Merged Gated Report

In [ ]:
import csv
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
report_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1"
expected = [
    "aggregate_method_bucket.csv",
    "selection_scores.csv",
    "selection_scores_feasible.csv",
    "constraint_summary.csv",
    "paired_bootstrap.csv",
    "aggregate_target_length.csv",
    "candidate_coverage_by_length.csv",
    "candidate_coverage_by_relation.csv",
    "gate_activation_summary.csv",
    "report_summary.json",
]
for name in expected:
    path = report_dir / name
    assert path.exists(), f"Missing report artifact: {path}"

with (report_dir / "report_summary.json").open() as f:
    payload = json.load(f)
assert payload["primary_paraphrase_bucket"] == "declarative_paraphrases"
assert payload["qa_format_generalization"] == "secondary_diagnostic"
assert payload["constraints_enabled"] is True
assert payload["analysis_500_used"] is False

with (report_dir / "selection_scores.csv").open(newline="") as f:
    scores = list(csv.DictReader(f))
with (report_dir / "selection_scores_feasible.csv").open(newline="") as f:
    feasible = list(csv.DictReader(f))
with (report_dir / "gate_activation_summary.csv").open(newline="") as f:
    gates = list(csv.DictReader(f))

required_methods = {
    "base",
    "target_logit_bias",
    "target_logit_bias_gated_subject",
    "target_candidate_insert",
    "target_candidate_insert_gated_subject",
    "prompt_memory",
    "prompt_memory_gated_subject",
    "prompt_memory_gated_subject_relation",
    "myopic_score",
    "myopic_score_gated_subject",
    "myopic_score_gated_subject_relation",
    "no_rollout_bridge",
    "no_rollout_bridge_gated_subject",
    "no_rollout_bridge_gated_subject_relation",
    "mc_bridge",
    "mc_bridge_gated_subject",
    "mc_bridge_gated_subject_relation",
    "raw_bridge_gated_subject",
}
observed = {row["method"] for row in scores}
missing = required_methods - observed
assert not missing, f"Missing methods in selection scores: {missing}"

print("selection rows:", len(scores))
print("feasible rows:", len(feasible))
print("gate activation rows:", len(gates))
print()
print("Feasible methods:")
for row in feasible:
    print(row)
print()
print("Gate activation summary:")
for row in gates:
    print(row)


## Expected Outcome

Use this report to decide the next dev step:

```text
If mc_bridge_gated_subject or mc_bridge_gated_subject_relation is feasible and competitive:
  proceed to matched-KL/guidance sweep.

If myopic_score_gated_subject dominates MC bridge:
  do matched-KL/guidance sweep before making a bridge claim.

If all gated methods lose too much rewrite/paraphrase activation:
  improve gate/span handling before analysis_500.
```

Do not run `analysis_500` from this notebook.
